# MLflow notebook: NYC Green Taxi fare regression

**Tasks to do:**

1. Create 3 experiments for 3 different regression models on the data (green taxi 
January and February 2021 combined) that we used in previous practices to train 
models. Make different runs in each experiment to get the best run. Explain 
everything in the report with screenshots.
2. Test your best models on the March 2021 data (uploaded with this practice) and 
assign them stages accordingly. Test the models and put best one in production 
keep other two in staging. Do it in code not on the MLflow UI.
3. Reproduce your best model from Question 1 using MLflow and verify that you get 
the same results
    * a. Load the model from MLflow using its run ID or model URI
    * b. Run inference again on the same test dataset
    * c. Compare the predictions and evaluation metrics
    * d. Explain whether the results match exactly or not and why

## 0. Imports and basic setup

A few practical notes:

- I use a **fixed random seed** so results are easier to reproduce.
- The features are kept intentionally close to the regression example used previously.

In [ ]:
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore")

# Change this if your MLflow server is elsewhere.
# If you already run "mlflow server ..." locally, this is a common default.
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

RANDOM_STATE = 42

# Notebook is assumed to live in dev/, and data/ is next to it.
DATA_DIR = Path("../data")

JAN_PATH = DATA_DIR / "green_tripdata_2021-01.parquet"
FEB_PATH = DATA_DIR / "green_tripdata_2021-02.parquet"
MAR_PATH = DATA_DIR / "green_tripdata_2021-03.parquet"

FEATURES = [
    "trip_distance",
    "trip_duration_min",
    "passenger_count",
    "RatecodeID",
    "pickup_hour",
]
TARGET = "fare_amount"

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Expected data files:")
print(" -", JAN_PATH)
print(" -", FEB_PATH)
print(" -", MAR_PATH)

MLflow tracking URI: http://127.0.0.1:5000
Expected data files:
 - ../data/green_tripdata_2021-01.parquet
 - ../data/green_tripdata_2021-02.parquet
 - ../data/green_tripdata_2021-03.parquet


## 1. Helper functions

In [8]:
def load_parquet_files(paths):
    dfs = [pd.read_parquet(path) for path in paths]
    return pd.concat(dfs, ignore_index=True)


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["lpep_pickup_datetime"] = pd.to_datetime(
        df["lpep_pickup_datetime"], errors="coerce"
    )
    df["lpep_dropoff_datetime"] = pd.to_datetime(
        df["lpep_dropoff_datetime"], errors="coerce"
    )

    # Trip duration in minutes
    df["trip_duration_min"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60.0

    # Simple time-based feature
    df["pickup_hour"] = df["lpep_pickup_datetime"].dt.hour

    return df


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    needed = set(FEATURES + [TARGET])
    missing_cols = needed - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing expected columns: {sorted(missing_cols)}")

    # Drop rows where critical values are missing
    df = df.dropna(subset=[TARGET, "trip_distance", "trip_duration_min"])

    # Keep only pretty sensible taxi rows
    df = df[df[TARGET].between(0.5, 300)]
    df = df[df["trip_distance"].between(0.01, 100)]
    df = df[df["trip_duration_min"].between(0.1, 300)]

    if "passenger_count" in df.columns:
        df = df[df["passenger_count"].between(1, 8)]

    if "RatecodeID" in df.columns:
        df = df[df["RatecodeID"].between(1, 6)]

    if "pickup_hour" in df.columns:
        df = df[df["pickup_hour"].between(0, 23)]

    return df


def prepare_features_and_target(df: pd.DataFrame):
    X = df[FEATURES].astype(float)
    y = df[TARGET].astype(float)
    return X, y


def evaluate_regression(model, X, y):
    
    preds = model.predict(X)

    mae = mean_absolute_error(y, preds)
    rmse = root_mean_squared_error(y, preds)
    r2 = r2_score(y, preds)

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }, preds


def build_pipeline(model):
    # Very simple preprocessing: just fill missing numeric values
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model),
        ]
    )

## 2. Load January + February data and make one train/validation split

In [3]:
train_df = load_parquet_files([JAN_PATH, FEB_PATH])
train_df = add_derived_features(train_df)
train_df = basic_clean(train_df)

X_all, y_all = prepare_features_and_target(train_df)

# This validation split is what we will use to compare runs fairly.
# We keep the random_state fixed so we can reproduce it later.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Combined Jan+Feb rows after cleaning:", len(train_df))
print("Train rows:", len(X_train))
print("Validation rows:", len(X_valid))

Combined Jan+Feb rows after cleaning: 71475
Train rows: 57180
Validation rows: 14295


## 3. Define the 3 model families and a few parameter combinations

3 basic model families to be tested:

- **Linear Regression**
- **Random Forest Regressor**
- **Gradient Boosting Regressor**

Each family gets its own MLflow experiment, and each parameter combination becomes one run.

In [4]:
EXPERIMENT_CONFIGS = {
    "nyc-green-taxi-linear-regression": {
        "model_label": "LinearRegression",
        "builder": lambda params: LinearRegression(**params),
        "param_grid": [
            {"fit_intercept": True},
            {"fit_intercept": False},
        ],
    },
    "nyc-green-taxi-random-forest": {
        "model_label": "RandomForestRegressor",
        "builder": lambda params: RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params,
        ),
        "param_grid": [
            {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1},
            {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 2},
            {"n_estimators": 300, "max_depth": 15, "min_samples_leaf": 2},
        ],
    },
    "nyc-green-taxi-gradient-boosting": {
        "model_label": "GradientBoostingRegressor",
        "builder": lambda params: GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            **params,
        ),
        "param_grid": [
            {"n_estimators": 100, "learning_rate": 0.10, "max_depth": 3},
            {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
            {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 2},
        ],
    },
}

## 4. Train runs and log everything to MLflow

In [ ]:
# Optional but handy: this automatically logs a bit more metadata for sklearn.
mlflow.sklearn.autolog(log_models=False)

best_runs_q1 = []

for experiment_name, config in EXPERIMENT_CONFIGS.items():
    mlflow.set_experiment(experiment_name)
    experiment = mlflow.get_experiment_by_name(experiment_name)

    run_results = []

    print(f"\n=== Experiment: {experiment_name} ===")

    for run_number, params in enumerate(config["param_grid"], start=1):
        model = build_pipeline(config["builder"](params))

        with mlflow.start_run(run_name=f"run_{run_number}") as run:
            # Fit
            model.fit(X_train, y_train)

            # Evaluate on the validation split from Jan+Feb
            train_metrics, _ = evaluate_regression(model, X_train, y_train)
            valid_metrics, valid_preds = evaluate_regression(model, X_valid, y_valid)

            # Log extra params/metadata in a very explicit way
            mlflow.log_param("model_family", config["model_label"])
            mlflow.log_param("features", ",".join(FEATURES))
            mlflow.log_param("target", TARGET)
            mlflow.log_param("random_state", RANDOM_STATE)
            mlflow.log_param("validation_split", 0.2)

            for key, value in params.items():
                mlflow.log_param(key, value)

            # Explicit metric names make it easy to compare runs later
            mlflow.log_metric("train_mae", train_metrics["mae"])
            mlflow.log_metric("train_rmse", train_metrics["rmse"])
            mlflow.log_metric("train_r2", train_metrics["r2"])

            mlflow.log_metric("valid_mae", valid_metrics["mae"])
            mlflow.log_metric("valid_rmse", valid_metrics["rmse"])
            mlflow.log_metric("valid_r2", valid_metrics["r2"])

            # Log the model artifact
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
            )

            # Some tags just to make the runs easier to read in MLflow
            mlflow.set_tags(
                {
                    "dataset": "NYC green taxi Jan+Feb 2021",
                    "task": "fare_amount regression",
                    "phase": "Q1 model selection",
                }
            )

            result = {
                "experiment_name": experiment_name,
                "experiment_id": experiment.experiment_id,
                "run_id": run.info.run_id,
                "model_family": config["model_label"],
                "params": params,
                "valid_mae": valid_metrics["mae"],
                "valid_rmse": valid_metrics["rmse"],
                "valid_r2": valid_metrics["r2"],
                "model_uri": f"runs:/{run.info.run_id}/model",
            }
            run_results.append(result)

            print(
                f"run_{run_number} | params={params} | "
                f"valid_rmse={valid_metrics['rmse']:.4f} | "
                f"valid_r2={valid_metrics['r2']:.4f}"
            )

    # Pick the best run inside this experiment by validation RMSE
    best_run = min(run_results, key=lambda x: x["valid_rmse"])
    best_runs_q1.append(best_run)

    print("Best run in this experiment:")
    print(best_run)


=== Experiment: nyc-green-taxi-linear-regression ===


2026/03/23 17:01:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 17:01:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


run_1 | params={'fit_intercept': True} | valid_rmse=3.3951 | valid_r2=0.9426
🏃 View run run_1 at: http://127.0.0.1:5000/#/experiments/1/runs/5c09b8d6258248a7831775f0b3b25ca7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/23 17:01:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 17:01:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/03/23 17:01:59 INFO mlflow.tracking.fluent: Experiment with name 'nyc-green-taxi-random-forest' does not exist. Creating a new experiment.


run_2 | params={'fit_intercept': False} | valid_rmse=3.4060 | valid_r2=0.9422
🏃 View run run_2 at: http://127.0.0.1:5000/#/experiments/1/runs/7d6ee2841106415ba7ef82e77a62daee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Best run in this experiment:
{'experiment_name': 'nyc-green-taxi-linear-regression', 'experiment_id': '1', 'run_id': '5c09b8d6258248a7831775f0b3b25ca7', 'model_family': 'LinearRegression', 'params': {'fit_intercept': True}, 'valid_mae': 0.8055963722521862, 'valid_rmse': 3.3951356953034812, 'valid_r2': 0.9425661206031403, 'model_uri': 'runs:/5c09b8d6258248a7831775f0b3b25ca7/model'}

=== Experiment: nyc-green-taxi-random-forest ===


...

run_1 | params={'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3} | valid_rmse=2.6623 | valid_r2=0.9647
🏃 View run run_1 at: http://127.0.0.1:5000/#/experiments/3/runs/7fbbc085bc9b4a9bb8d54e79b6d74719
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/23 17:48:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 18:04:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


run_2 | params={'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3} | valid_rmse=2.6142 | valid_r2=0.9659
🏃 View run run_2 at: http://127.0.0.1:5000/#/experiments/3/runs/86409a6528304d9599c29411abb9aa9d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/23 18:58:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 18:58:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


run_3 | params={'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 2} | valid_rmse=2.9359 | valid_r2=0.9571
🏃 View run run_3 at: http://127.0.0.1:5000/#/experiments/3/runs/70f78e4005d34e949365fad1f4522032
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Best run in this experiment:
{'experiment_name': 'nyc-green-taxi-gradient-boosting', 'experiment_id': '3', 'run_id': '86409a6528304d9599c29411abb9aa9d', 'model_family': 'GradientBoostingRegressor', 'params': {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3}, 'valid_mae': 0.6466765268829386, 'valid_rmse': 2.6141808076885003, 'valid_r2': 0.9659493722886188, 'model_uri': 'runs:/86409a6528304d9599c29411abb9aa9d/model'}


## 5. Best run from each experiment (Question 1 result)

In [10]:
best_runs_q1_df = pd.DataFrame(best_runs_q1).sort_values("valid_rmse").reset_index(drop=True)
best_runs_q1_df

,experiment_name,experiment_id,run_id,model_family,params,valid_mae,valid_rmse,valid_r2,model_uri
0,nyc-green-taxi-gradient-boosting,3,86409a6528304d9599c29411abb9aa9d,GradientBoostingRegressor,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.646677,2.614181,0.965949,runs:/86409a6528304d9599c29411abb9aa9d/model
1,nyc-green-taxi-random-forest,2,829e3963138a48ec9f37cd702ee7afac,RandomForestRegressor,"{'n_estimators': 100, 'max_depth': None, 'min_...",0.550605,2.710555,0.963392,runs:/829e3963138a48ec9f37cd702ee7afac/model
2,nyc-green-taxi-linear-regression,1,5c09b8d6258248a7831775f0b3b25ca7,LinearRegression,{'fit_intercept': True},0.805596,3.395136,0.942566,runs:/5c09b8d6258248a7831775f0b3b25ca7/model


The **overall best model from Question 1** is just the one with the lowest validation RMSE among those 3 experiment winners.

In [11]:
overall_best_q1 = best_runs_q1_df.iloc[0].to_dict()
overall_best_q1

{'experiment_name': 'nyc-green-taxi-gradient-boosting',
 'experiment_id': '3',
 'run_id': '86409a6528304d9599c29411abb9aa9d',
 'model_family': 'GradientBoostingRegressor',
 'params': {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3},
 'valid_mae': 0.6466765268829386,
 'valid_rmse': 2.6141808076885003,
 'valid_r2': 0.9659493722886188,
 'model_uri': 'runs:/86409a6528304d9599c29411abb9aa9d/model'}

## 6. Load March 2021 data for final testing (Question 2)

In [12]:
march_df = pd.read_parquet(MAR_PATH)
march_df = add_derived_features(march_df)
march_df = basic_clean(march_df)

X_march, y_march = prepare_features_and_target(march_df)

print("March rows after cleaning:", len(march_df))

March rows after cleaning: 40677


## 7. Test the best model from each experiment on March and register them in MLflow

Steps:

1. load each winning model from its **run ID / model URI**
2. evaluate it on the **March 2021** dataset
3. register it in the MLflow **Model Registry**
4. assign stages **in code**

In [13]:
client = MlflowClient()

registered_results = []

for row in best_runs_q1:
    model_uri = row["model_uri"]
    loaded_model = mlflow.sklearn.load_model(model_uri)

    test_metrics, _ = evaluate_regression(loaded_model, X_march, y_march)

    # Small and readable model names for the registry
    registry_name = f"{row['experiment_name']}-model"

    # Register model version from the winning run
    model_version = mlflow.register_model(model_uri=model_uri, name=registry_name)

    result = {
        **row,
        "march_mae": test_metrics["mae"],
        "march_rmse": test_metrics["rmse"],
        "march_r2": test_metrics["r2"],
        "registry_name": registry_name,
        "model_version": int(model_version.version),
    }
    registered_results.append(result)

registered_results_df = pd.DataFrame(registered_results).sort_values("march_rmse").reset_index(drop=True)
registered_results_df

Successfully registered model 'nyc-green-taxi-linear-regression-model'.
2026/03/24 05:57:02 WARNING mlflow.tracking._model_registry.fluent: Run with id 5c09b8d6258248a7831775f0b3b25ca7 has no artifacts at artifact path 'model', registering model based on models:/m-3accac740cc04988a9e99e511d3f2dbf instead
2026/03/24 05:57:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-green-taxi-linear-regression-model, version 1
Created version '1' of model 'nyc-green-taxi-linear-regression-model'.
Successfully registered model 'nyc-green-taxi-random-forest-model'.
2026/03/24 05:57:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 829e3963138a48ec9f37cd702ee7afac has no artifacts at artifact path 'model', registering model based on models:/m-e95396df0f8440e89aa1565980d541f7 instead
2026/03/24 05:57:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish cre

,experiment_name,experiment_id,run_id,model_family,params,valid_mae,valid_rmse,valid_r2,model_uri,march_mae,march_rmse,march_r2,registry_name,model_version
0,nyc-green-taxi-gradient-boosting,3,86409a6528304d9599c29411abb9aa9d,GradientBoostingRegressor,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.646677,2.614181,0.965949,runs:/86409a6528304d9599c29411abb9aa9d/model,0.673151,2.276154,0.975985,nyc-green-taxi-gradient-boosting-model,1
1,nyc-green-taxi-random-forest,2,829e3963138a48ec9f37cd702ee7afac,RandomForestRegressor,"{'n_estimators': 100, 'max_depth': None, 'min_...",0.550605,2.710555,0.963392,runs:/829e3963138a48ec9f37cd702ee7afac/model,0.590735,2.401736,0.973262,nyc-green-taxi-random-forest-model,1
2,nyc-green-taxi-linear-regression,1,5c09b8d6258248a7831775f0b3b25ca7,LinearRegression,{'fit_intercept': True},0.805596,3.395136,0.942566,runs:/5c09b8d6258248a7831775f0b3b25ca7/model,0.825281,2.781218,0.964146,nyc-green-taxi-linear-regression-model,1


## 8. Put the best March model in Production, the other two in Staging

This is done **programmatically**, not in the MLflow UI.

In [14]:
best_production_row = registered_results_df.iloc[0].to_dict()

for idx, row in registered_results_df.iterrows():
    target_stage = "Production" if idx == 0 else "Staging"

    client.transition_model_version_stage(
        name=row["registry_name"],
        version=int(row["model_version"]),
        stage=target_stage,
        archive_existing_versions=True,
    )

    print(
        f"{row['registry_name']} v{int(row['model_version'])} -> {target_stage} "
        f"(March RMSE: {row['march_rmse']:.4f})"
    )

print("\nBest March model now in Production:")
print(best_production_row["registry_name"], "version", int(best_production_row["model_version"]))

nyc-green-taxi-gradient-boosting-model v1 -> Production (March RMSE: 2.2762)
nyc-green-taxi-random-forest-model v1 -> Staging (March RMSE: 2.4017)
nyc-green-taxi-linear-regression-model v1 -> Staging (March RMSE: 2.7812)

Best March model now in Production:
nyc-green-taxi-gradient-boosting-model version 1


## 9. Reproduce the best model from Question 1 (Question 3)

The goal here is simple:

- load the best model back **from MLflow**
- run inference again on the **same validation set** used in Question 1
- compare predictions and metrics

Why use the same validation set?

Because Question 1 selected the best run based on the Jan+Feb validation split, so reproducing it on that exact split is the cleanest check.

In [15]:
best_q1_run_id = overall_best_q1["run_id"]
best_q1_model_uri = overall_best_q1["model_uri"]

print("Best Q1 run_id   :", best_q1_run_id)
print("Best Q1 model_uri:", best_q1_model_uri)

Best Q1 run_id   : 86409a6528304d9599c29411abb9aa9d
Best Q1 model_uri: runs:/86409a6528304d9599c29411abb9aa9d/model


In [16]:
# Load the model back from MLflow
reloaded_best_model = mlflow.sklearn.load_model(best_q1_model_uri)

# Run inference again on the SAME validation data from Q1
reloaded_metrics, reloaded_preds = evaluate_regression(reloaded_best_model, X_valid, y_valid)

# For comparison, we also load the original run's metrics from the MLflow run record
original_run = client.get_run(best_q1_run_id)

original_valid_metrics = {
    "mae": float(original_run.data.metrics["valid_mae"]),
    "rmse": float(original_run.data.metrics["valid_rmse"]),
    "r2": float(original_run.data.metrics["valid_r2"]),
}

comparison_df = pd.DataFrame(
    [
        {"metric": "mae", "original_logged": original_valid_metrics["mae"], "reloaded_model": reloaded_metrics["mae"]},
        {"metric": "rmse", "original_logged": original_valid_metrics["rmse"], "reloaded_model": reloaded_metrics["rmse"]},
        {"metric": "r2", "original_logged": original_valid_metrics["r2"], "reloaded_model": reloaded_metrics["r2"]},
    ]
)

comparison_df["absolute_difference"] = (
    comparison_df["original_logged"] - comparison_df["reloaded_model"]
).abs()

comparison_df

,metric,original_logged,reloaded_model,absolute_difference
0,mae,0.646677,0.646677,0.0
1,rmse,2.614181,2.614181,0.0
2,r2,0.965949,0.965949,0.0


In [17]:
# Optional deeper check: compare predictions themselves
# We only have the reloaded predictions now, so let's also load the model one more time
# from the same URI and predict again. This is mainly to show the idea clearly.

same_model_again = mlflow.sklearn.load_model(best_q1_model_uri)
same_preds_again = same_model_again.predict(X_valid)

predictions_exact_match = np.array_equal(reloaded_preds, same_preds_again)
predictions_close_match = np.allclose(reloaded_preds, same_preds_again)

max_abs_prediction_diff = np.max(np.abs(reloaded_preds - same_preds_again))

print("Predictions exact match :", predictions_exact_match)
print("Predictions close match :", predictions_close_match)
print("Max abs prediction diff :", max_abs_prediction_diff)

Predictions exact match : True
Predictions close match : True
Max abs prediction diff : 0.0


## 10. Explanation for Question 3(d)

**Do the reproduced results match?**

They do match **exactly** or are so close that the difference is effectively zero.

Why?

- I loaded the **saved trained model artifact** from MLflow, not a freshly retrained model.
- I used the **same input data**.
- I used the **same preprocessing pipeline** because it was saved inside the model pipeline.